In [ ]:
import pandas as pd

df1 = pd.read_csv('Appointments.csv')
df2 = pd.read_csv('Appointments(1).csv')
df3 = pd.read_csv('Appointments(2).csv')
df4 = pd.read_csv('Appointments(3).csv')
df5 = pd.read_csv('Appointments(4).csv')
df6 = pd.read_csv('Appointments(5).csv')
df7 = pd.read_csv('Appointments(6).csv')

df = pd.concat([df1,df2,df3,df4,df5,df6,df7], ignore_index=True)
df = df.drop_duplicates(subset=['Appointment Date', 'Booked Date', 'Invoice No', 'Guest Name',
       'Service Name', 'Center Name', 'Start Time', 'End Time',
       'Scheduled Service Duration', 'Scheduled Service and Recovery Duration',
       'Recovery Time', 'Stylist', 'Status'])

df["Appointment Date"] = pd.to_datetime(df["Appointment Date"])
df["Booked Date"] = pd.to_datetime(df["Booked Date"])
df = df.sort_values("Appointment Date")

df



In [ ]:
df['Guest Name'] = df["Guest Name"].str.strip().str.title()
df = df[df["Status"] == "Closed"]
df = df[df["Guest Name"] != "Online Guest"]
print(df.shape)
print(df["Guest Name"].nunique())
print(df["Appointment Date"].min())
print(df["Appointment Date"].max())

In [ ]:
df.to_csv("appointments_clean.csv", index=False)
df = pd.read_csv("appointments_clean.csv")
df

In [ ]:
visits = df.drop_duplicates(subset=["Guest Name", "Appointment Date"])
visits["Appointment Date"] = pd.to_datetime(visits["Appointment Date"])
visits['Gap'] = visits.groupby("Guest Name")["Appointment Date"].diff()
visits["Gap"] = visits["Gap"].dt.days
visits

In [ ]:
guest_summary = visits.groupby("Guest Name").agg(
    AverageGap = ('Gap', "mean"),
    LastVisit = ("Appointment Date", "max"),
    VisitCount = ("Appointment Date", "count")
)

guest_summary['DaysSinceLastVisit'] = pd.Timestamp("today") - guest_summary['LastVisit']
guest_summary["DaysSinceLastVisit"] = guest_summary["DaysSinceLastVisit"].dt.days
guest_summary

In [ ]:
def classify(row):
    if row["VisitCount"] == 1:
        # single visit logic
        if row["DaysSinceLastVisit"] < 60:
            return "Active"
        elif 60 <= row["DaysSinceLastVisit"]<120:
            return "At Risk"
        elif row["DaysSinceLastVisit"] >= 120:
            return "Churned"
    else:
        # multi visit logic
        if row["DaysSinceLastVisit"] < row["AverageGap"]:
            return "Active"
        elif row["AverageGap"] <= row["DaysSinceLastVisit"]< 2*row["AverageGap"]:
            return "At Risk"
        elif row["DaysSinceLastVisit"] >= 2 * row["AverageGap"]:
            return "Churned"

guest_summary["Status"] = guest_summary.apply(classify, axis=1)
print(guest_summary["Status"].value_counts())
guest_summary